# BICA v0.3 — адаптивная визуальная саморефлексия

**Цель:** сравнить исходную Qwen2.5-VL с фиксированным агентом `reflect` и новым `reflect_adaptive` на одинаковых изображениях. Второй просмотр изолирован от первой интерпретации. При расхождении ответов агент делает дополнительный визуальный challenge по оригинальному изображению и одному фрагменту.

**Сейчас:** только технический тест на двух спорных отладочных изображениях (004/005), **не** измерение научной точности. Человеческие эталоны и 4 уровня подсказок A–D ещё не утверждены.

Порядок: **1. T4 GPU → 2. Установка → перезапуск сеанса при необходимости → 3. Проверка → 4–8. Пилот → 9. Результаты → 10. Скачать ZIP.**

Встроенный архив содержит код и 6 изображений. Старые папки v0.2/v0.2.1 не перезаписываются. GPU не требуется для установки и Python-тестов, но необходим для Qwen.

## 1. Проверка GPU

In [ ]:
import sys, platform
import torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU не обнаружен. Выберите Среда выполнения → Сменить среду выполнения → T4 GPU, подключитесь снова.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

## 2. Библиотеки (однократно)
Если библиотека меняется — после этой ячейки **перезапустите сеанс**, затем начните с раздела 3. Не переустанавливайте Pillow поверх загруженного импорта.

In [ ]:
import sys, os, subprocess, json
from pathlib import Path
from importlib import metadata

PIN = {'transformers': '4.57.1', 'accelerate': '1.11.0',
       'huggingface-hub': '0.36.2', 'Pillow': '11.3.0'}


def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None


def pillow_subprocess_check():
    result = subprocess.run([sys.executable, '-c',
        'from PIL import Image, ImageOps; print(Image.__version__)'],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode:
        print('Pillow preflight:', result.stderr.strip().splitlines()[-1:])
    return result.returncode == 0

needs = {pkg: ver for pkg, ver in PIN.items() if installed_version(pkg) != ver}
pillow_broken = not pillow_subprocess_check()
if needs or pillow_broken:
    print('Изменяемые зависимости:', needs or {'Pillow':'восстановление импорта'})
    # Эти библиотеки не используются агентом; при hub<1 конфликтуют с окружением Colab.
    existing_unused = [x for x in ('gradio', 'gradio-client', 'diffusers') if installed_version(x)]
    if existing_unused:
        print('Удаляем неиспользуемые конфликтующие пакеты:', existing_unused)
        subprocess.run([sys.executable, '-m','pip','uninstall','-y', *existing_unused], check=True)
    # Установка Pillow отдельно исключает неверную смесь его .py и бинарных модулей.
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','Pillow'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--no-deps',
                    'Pillow==11.3.0'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir',
                    'transformers==4.57.1', 'accelerate==1.11.0',
                    'huggingface-hub==0.36.2'], check=True)
    Path('/content/bica_restart_required.pid').write_text(str(os.getpid()))
    print('\nГОТОВО. ВАЖНО: сейчас нажмите «Среда выполнения → Перезапустить сеанс».')
    print('После перезапуска начинайте с ЯЧЕЙКИ 3. Не запускайте установку повторно.')
else:
    print('Нужные версии уже установлены; импорт Pillow из нового процесса исправен.')
    print('Можно идти к ячейке 3 без повторной установки.')

## 3. Проверка после установки и перезапуска

In [ ]:
import os, sys
from pathlib import Path
marker = Path('/content/bica_restart_required.pid')
if marker.exists() and marker.read_text().strip() == str(os.getpid()):
    raise RuntimeError('Установка завершена, но сеанс не перезапущен. Среда выполнения → Перезапустить сеанс; затем выполните ячейку 3.')

import PIL, torch, transformers, accelerate, huggingface_hub
from PIL import Image, ImageOps
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

expected = {'Pillow': '11.3.0', 'Transformers': '4.57.1',
            'Accelerate': '1.11.0', 'Hugging Face Hub': '0.36.2'}
actual = {'Pillow': PIL.__version__, 'Transformers': transformers.__version__,
          'Accelerate': accelerate.__version__, 'Hugging Face Hub': huggingface_hub.__version__}
for name, version in actual.items():
    print(name + ':', version)
    assert version == expected[name], f'{name}: ожидали {expected[name]}, получили {version}'
assert torch.cuda.is_available(), 'Нет GPU: выберите T4 GPU и переподключитесь.'
print('GPU:', torch.cuda.get_device_name(0))
print('Pillow.Image and ImageOps: OK; Qwen2.5-VL imports: OK')

## 4. Восстановление полного кода из блокнота
В этом блокноте нет необходимости загружать отдельный ZIP. Внутри находятся Python-модули, тесты и отладочные изображения.

In [ ]:
from pathlib import Path
import base64, io, zipfile, hashlib, sys, os
EMBEDDED_B64 = (
    '...'
    'cHlQSwUGAAAAAD0APQBrFAAAMjsGAAAA'
)
raw = base64.b64decode(EMBEDDED_B64)
expected_sha = '26dc63cb5b3c0b166fab34e02921776315f2ce9d63f2d3a8420d3df33c4f100a'
assert hashlib.sha256(raw).hexdigest() == expected_sha, 'Повреждён встроенный архив проекта.'
TARGET = Path('/content/BICA_v030_workspace')
ROOT = TARGET / 'bica-main'
marker = ROOT/'.embedded_archive_sha256'
if not ROOT.exists():
    TARGET.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        for name in z.namelist():
            p = Path(name)
            assert not p.is_absolute() and '..' not in p.parts, 'Недопустимое имя в ZIP'
        z.extractall(TARGET)
    marker.write_text(expected_sha)
else:
    assert marker.exists() and marker.read_text().strip() == expected_sha, 'Другая сборка BICA в этой среде — используйте новую среду Colab или новую папку.'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MANIFEST = ROOT/'data'/'ambibench_debug'/'manifest.jsonl'
REGRESSION = ROOT/'data'/'ambibench_debug'/'manifest_regression_v021.jsonl'
print('BICA v0.3:', ROOT)
print('Отладочные изображения:', len(list((ROOT/'data'/'ambibench_debug'/'images').iterdir())))

## 5. Автоматические тесты — без GPU-запросов
Проверяются схема, независимость второго просмотра, выбор challenge, ограничения бюджета, экспорт и антиутечка эталонов.

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m','unittest','discover','-s','tests','-q'],
                        cwd=ROOT, capture_output=True, text=True)
print('\n'.join((result.stdout+'\n'+result.stderr).splitlines()[-18:]))
assert result.returncode == 0, 'Проверки Python не прошли — GPU запускать нельзя'
print('Автоматические тесты пройдены.')

## 6. Папка результатов (отдельно от v0.2.1)
Drive сохраняет трассировки между сессиями; повторное исполнение с той же конфигурацией пропускает готовые случаи. Если Drive не нужен, поставьте `SAVE_TO_DRIVE=False` и **скачайте ZIP до отключения среды**.

In [ ]:
from pathlib import Path
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE = Path('/content/drive/MyDrive/BICA_ReflectiveVLM_v030')
else:
    SAVE = Path('/content/BICA_ReflectiveVLM_v030')
SAVE.mkdir(parents=True, exist_ok=True)
print('Результаты:', SAVE)

## 7. Qwen2.5-VL-3B на T4
Фиксируем ту же ревизию модели, что и в v0.2.1. Первый запуск скачивает веса. Убедитесь, что выбрана **T4 GPU**.

In [ ]:
from illusion_agent.colab import QwenClient
import json
MODEL_REVISION = '66285546d2b821cf421d4f5eb2576359d3770cd3'
client = QwenClient(model_id='Qwen/Qwen2.5-VL-3B-Instruct',
                    revision=MODEL_REVISION, max_tokens=768)
print(json.dumps(client.metadata, ensure_ascii=False, indent=2))

## 8. Первый реальный прогон — 2 изображения × 3 режима

Сравниваются `direct`, `reflect` и **`reflect_adaptive`**. Будет 6 случаев, 14–16 основных вызовов Qwen (по решению адаптивного алгоритма) + до 6 вызовов исправления JSON. `max_calls=24` — жёсткий предел, не ожидаемый расход.

Это только диагностика. Исходные метки отладочных изображений не утверждены и не используются в инференсе.

In [ ]:
from illusion_agent.colab import run_pilot
from illusion_agent.diagnostics import analyze_run
from collections import Counter
PILOT_DIR = SAVE / 'regression_004_005_direct_reflect_adaptive_v030'
summary = run_pilot(client, REGRESSION, PILOT_DIR,
                    modes=('direct','reflect','reflect_adaptive'),
                    max_calls=24,seed=42,schema_repair=True)
print(json.dumps(summary, ensure_ascii=False, indent=2))
report=analyze_run(PILOT_DIR)
print(json.dumps(report, ensure_ascii=False, indent=2))

## 9. Проверить ответы, триггеры и количество вызовов
Не считайте количество названных предметов оценкой точности: `duck`, `bird`, `animal` могут описывать один объект.

In [ ]:
import json
import pandas as pd
rows=[json.loads(s) for s in (PILOT_DIR/'results.jsonl').read_text().splitlines() if s.strip()]
events=[json.loads(s) for s in (PILOT_DIR/'trace.jsonl').read_text().splitlines() if s.strip()]
flags={e['case_id']:e for e in events if e['event']=='adaptive_decision'}
display(pd.DataFrame([{
    'case':r['case_id'], 'image':r['image_id'], 'mode':r['mode'], 'status':r['status'],
    'labels':'; '.join(x['label']+' ('+x['status']+')' for x in r.get('answer',{}).get('interpretations',[])),
    'adaptive_challenge':flags.get(r['case_id'],{}).get('need_challenge','—'),
    'trigger':', '.join(flags.get(r['case_id'],{}).get('reasons',[])),
    'error':r.get('detail','')
} for r in rows]))
print('Случаи с ошибками:',sum(r['status']!='ok' for r in rows))
print('Результаты сгенерированы реально Qwen:',not summary['synthetic'])

## 10. Необязательный полный отладочный прогон — 6 изображений

**По умолчанию отключён.** 18 случаев, до 48 основных вызовов плюс до 18 вызовов восстановления формата. Эта выборка *без отрицательных контролей* не подходит для заявления о приросте научной точности.

In [ ]:
RUN_FULL_DEBUG = False
FULL_DIR = SAVE / 'pilot_six_images_v030'
if RUN_FULL_DEBUG:
    full = run_pilot(client, MANIFEST, FULL_DIR,
                     modes=('direct','reflect','reflect_adaptive'),
                     max_calls=75, seed=42, schema_repair=True)
    print(json.dumps(full,ensure_ascii=False,indent=2))
    print(json.dumps(analyze_run(FULL_DIR),ensure_ascii=False,indent=2))
else:
    print('Полный отладочный прогон выключен — короткий прогон сохранён.')

## 11. Человеческий контроль эталонов до четырёх подсказок A–D

В проекте есть `tools.create_gold_forms` для подготовки **двух независимых форм**. Загрузите их команде и не раскрывайте исходные ответы до проверки. В A–D-пилоте на утверждённой выборке нужно также добавить обычные изображения без скрытой второй фигуры.

Генератор `tools.build_staged_manifest` **откажется** работать до разметки со статусом `adjudicated`, это намеренное ограничение.

In [ ]:
from tools.create_gold_forms import build as build_gold_review
GOLD_PACK=SAVE/'gold_review_v030'
if not GOLD_PACK.exists():
    count = build_gold_review(MANIFEST, GOLD_PACK)
    print('Созданы независимые формы по',count,'изображениям')
else:
    print('Формы уже есть, не перезаписываем:',GOLD_PACK)
print('Приватная таблица соответствий находится РЯДОМ с папкой, не внутри:', GOLD_PACK.parent/(GOLD_PACK.name+'_OPERATOR_ONLY_mapping.json'))

## 12. Режим A–D для утверждённых эталонов (отключён)

Если команда подготовила **адъюдицированный** `gold_adjudicated.jsonl` (с контролями, источниками, доказательствами и правильными/ложными гипотезами), вставьте его в проект и включите `RUN_ABCD=True`. В этой ячейке не появятся искусственные оценки качества.

Полный запуск требует много GPU-вызовов; ниже можно выбрать разделение dev/test и заранее проверить бюджет. Для тестового корпуса используйте отдельную папку и не изменяйте промпты после dev.

In [ ]:
RUN_ABCD = False  # ТОЛЬКО после ручной адъюдикации эталонов
SPLIT_ABCD = 'dev'  # сначала dev; test — отдельный окончательный запуск
GOLD_FILE = ROOT/'data'/'ambibench_debug'/'gold_adjudicated.jsonl'
if RUN_ABCD:
    from tools.build_staged_manifest import build as build_staged
    from illusion_agent.core import load_manifest
    staged = ROOT/'data'/'ambibench_debug'/'manifest_ABCD_v030.jsonl'
    if not staged.exists():
        build_staged(MANIFEST, GOLD_FILE, staged)
    selected = [r for r in load_manifest(staged) if r['split'] == SPLIT_ABCD]
    if not selected:
        raise ValueError('Нет утверждённых изображений для split=' + SPLIT_ABCD)
    modes = ('direct','reflect','reflect_adaptive')
    base_calls=sum(sum([1,3,4]) * len(r['cues']) for r in selected)
    safe_max_calls=base_calls + len(selected)*len(modes)*4  # до одного исправления JSON на случай
    print('Изображений:',len(selected),'случаев:',len(selected)*len(modes)*4,
          'базовый верхний предел вызовов:',base_calls,
          'резерв с восстановлением JSON:',safe_max_calls)
    CONFIRM_GPU_BUDGET=False
    if not CONFIRM_GPU_BUDGET:
        print('A–D НЕ запущен. Проверьте объём, затем выставьте CONFIRM_GPU_BUDGET=True.')
    else:
        expdir=SAVE/('adjudicated_ABCD_'+SPLIT_ABCD+'_v030')
        result=run_pilot(client,staged,expdir,modes=modes,
                         max_calls=safe_max_calls, seed=42,
                         schema_repair=True,split=SPLIT_ABCD)
        print(json.dumps(result,ensure_ascii=False,indent=2))
else:
    print('A–D пока не выполнялся: нужны два оценщика и утверждённые обычные контроли.')

## 13. Скачать результаты
Результаты v0.3 сохраняются отдельно от v0.2.1. Отправьте ZIP в чат для разбора разницы режимов и ошибок.

In [ ]:
from google.colab import files
import shutil
archive = shutil.make_archive('/content/BICA_v030_Results','zip',root_dir=str(SAVE))
print('Готово:',archive)
files.download(archive)

## Как интерпретировать

- `status=ok` означает только валидный JSON и завершение вычисления.
- `adaptive_challenge=True` означает решение о дополнительном визуальном анализе, не доказательство неоднозначности.
- Исходные тестовые изображения 004 и 005 используются **только для отладки**, их эталоны не утверждены.
- Реальная оценка требует разметки двух независимых участников, ordinary-контролей, всех условий A–D и парного сравнения.

Полная инструкция и научные ограничения: `docs/08_v030_protocol_ru.md`.